API for ACS

In [22]:
# import necessary packages
import pandas as pd
import requests
import time
import sqlite3 as sql
import numpy as np
import matplotlib.pyplot as plt

In [23]:
def read_key(keyfile): # Extract API key from txt file
    with open(keyfile) as f:
        return f.readline().strip("\n")

key = read_key('/Users/avicaballero/Desktop/STA 141B/acskey.txt')

In [24]:
years = range(2010, 2024) # range of years for ACS data
year_dfs = [] # empty list to contain tables from all years
for year in years:
    time.sleep(0.05) # time between data rqs
    url = f'https://api.census.gov/data/{year}/acs/acs5/profile'
    response = requests.get(url, 
    params={ # pulls area, median income, number people w bachelors, poverty rate, unemployment rate, and population
        "get": "NAME,DP03_0062E,DP02_0059E,DP03_0128PE,DP03_0009PE,DP05_0001E",
        "for": "metropolitan statistical area/micropolitan statistical area:*",
        "key": key
    })
    response.raise_for_status() # the first row are the headers, need to update that in df
    data = pd.DataFrame(response.json()[1:], columns=response.json()[0])
    data['year'] = year
    year_dfs.append(data)

In [25]:
acs_df = pd.concat(year_dfs, ignore_index=True) # join together dfs of all years

# rename the variables from codebook
acs_df = acs_df.rename(columns={'DP03_0062E': 'Med_Income', 'DP02_0059E': 'Bachelors', 'DP03_0128PE': 'Poverty_Rate', 
                        'DP03_0009PE': 'Unemployment_Rate', 'DP05_0001E':'Population',
                        'year':'Year'})
acs_df.tail()

,NAME,Med_Income,Bachelors,Poverty_Rate,Unemployment_Rate,Population,metropolitan statistical area/micropolitan statistical area,Year
13172,"Youngstown-Warren, OH Metro Area",55357,306958,17.6,5.5,428430,49660,2023
13173,"Yuba City, CA Metro Area",74762,118206,15.0,7.1,182050,49700,2023
13174,"Yuma, AZ Metro Area",60417,134669,16.5,8.1,207685,49740,2023
13175,"Zanesville, OH Micro Area",59203,59220,15.6,4.6,86382,49780,2023
13176,"Zapata, TX Micro Area",36527,7944,37.4,8.0,13855,49820,2023


In [26]:
 # in order to match formatting for metro area in zillow dataset, need to do string manipulation on NAME
# this separates the string, and rejoins the area in city, ST format
# keep the column area type to filter to metro areas only

parts = acs_df['NAME'].str.split(', ', expand=True)
state_area_type = parts[1].str.split(' ', expand=True)
acs_df['City'] = parts[0]
acs_df['State'] = state_area_type[0]
acs_df['Area_Type'] = state_area_type[1]
acs_df['RegionName'] = acs_df['City'] + ', ' + acs_df['State']

In [27]:
metro_df = acs_df[acs_df['Area_Type']=='Metro']


In [29]:
# subset the sf to only columns we need, drop NAs
metro_subset = metro_df[['Med_Income','Bachelors', 'Poverty_Rate', 'Unemployment_Rate', 'Population', 'Year', 'RegionName', 'metropolitan statistical area/micropolitan statistical area']]
clean_acs = metro_subset.copy()
print(clean_acs.isna().sum()) # check NA values
clean_acs = clean_acs.dropna()
print(clean_acs.isna().sum()) # ensure NA values are dropped

Med_Income                                                       0
Bachelors                                                      104
Poverty_Rate                                                     0
Unemployment_Rate                                                0
Population                                                       0
Year                                                             0
RegionName                                                       0
metropolitan statistical area/micropolitan statistical area      0
dtype: int64
Med_Income                                                     0
Bachelors                                                      0
Poverty_Rate                                                   0
Unemployment_Rate                                              0
Population                                                     0
Year                                                           0
RegionName                                                   

In [30]:
# load zillow dataset
zhvi = pd.read_csv('/Users/avicaballero/Desktop/STA 141B/zhvi_index.csv')
zhvi.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,2000-01-31,2000-02-29,2000-03-31,2000-04-30,2000-05-31,...,2025-04-30,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31
0,102001,0,United States,country,NaN,120245.401273,120456.951481,120719.304889,121282.001451,121929.597731,...,356520.996883,355847.084537,355240.355926,354820.996430,354597.242836,354834.693617,355261.536448,355925.870866,356703.081920,357444.592267
1,394913,1,"New York, NY",msa,NY,216213.074441,217131.858016,218059.151183,219938.205956,221884.032245,...,684527.238403,686280.002733,687742.249805,688904.163124,689687.004711,691074.903618,693389.288933,696701.313817,700090.397649,703126.246109
2,753899,2,"Los Angeles, CA",msa,CA,219357.633964,220173.922681,221261.211467,223424.550904,225790.566038,...,943189.059655,938463.706961,934153.202642,931587.011384,930526.406111,931572.591632,933964.652130,937248.897282,941169.205188,943673.884613
3,394463,3,"Chicago, IL",msa,IL,149975.869412,150114.703378,150379.115111,151036.905792,151828.148861,...,324141.432142,324425.469135,324778.263087,325598.611099,326571.864936,327874.770171,329150.068272,330662.715073,332355.926815,333939.343216
4,394514,4,"Dallas, TX",msa,TX,126453.868825,126510.191820,126574.940868,126743.087346,126964.783873,...,367625.466282,365231.672338,362889.504974,360891.495428,359541.993782,358900.902631,358555.530235,358294.304119,358078.600817,357649.079207


In [31]:
# ── Reshape Wide → Long ──────────────────────────────────────────────────────
# Raw CSV: one row per metro, one column per month.
# We melt into (RegionName, Date, ZHVI) long format for easier analysis.

meta_cols = ['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName']
date_cols = [c for c in zhvi.columns if c not in meta_cols]

zhvi_long = zhvi.melt(
    id_vars=meta_cols,
    value_vars=date_cols,
    var_name='Date',
    value_name='ZHVI'
)

zhvi_long['Date'] = pd.to_datetime(zhvi_long['Date'])
zhvi_long = zhvi_long.dropna(subset=['ZHVI'])

# Keep only metro-level rows (RegionType == 'msa')
zhvi_metro = zhvi_long[zhvi_long['RegionType'] == 'msa'].copy()
zhvi_metro = zhvi_metro.sort_values(['RegionName', 'Date']).reset_index(drop=True)

print(f'Total metro rows : {len(zhvi_metro):,}')
print(f'Unique metros    : {zhvi_metro["RegionName"].nunique()}')
print(f'Date range       : {zhvi_metro["Date"].min().date()} to {zhvi_metro["Date"].max().date()}')
zhvi_metro.head()

Total metro rows : 230,556
Unique metros    : 894
Date range       : 2000-01-31 to 2026-01-31


,RegionID,SizeRank,RegionName,RegionType,StateName,Date,ZHVI
0,394297,677,"Aberdeen, SD",msa,SD,2009-02-28,125066.417884
1,394297,677,"Aberdeen, SD",msa,SD,2009-03-31,125079.142534
2,394297,677,"Aberdeen, SD",msa,SD,2009-04-30,124888.758875
3,394297,677,"Aberdeen, SD",msa,SD,2009-05-31,124780.972274
4,394297,677,"Aberdeen, SD",msa,SD,2009-06-30,124593.031335


In [34]:
zhvi_metro['Year'] = zhvi_metro['Date'].dt.year
merged_df = pd.merge(zhvi_metro, clean_acs, on=['Year','RegionName'])

In [36]:
merged_df.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,Date,ZHVI,Year,Med_Income,Bachelors,Poverty_Rate,Unemployment_Rate,Population,metropolitan statistical area/micropolitan statistical area
0,394299,251,"Abilene, TX",msa,TX,2010-01-31,106606.410765,2010,42286,7112,15.8,5.6,163092,10180
1,394299,251,"Abilene, TX",msa,TX,2010-02-28,106885.973781,2010,42286,7112,15.8,5.6,163092,10180
2,394299,251,"Abilene, TX",msa,TX,2010-03-31,107120.321341,2010,42286,7112,15.8,5.6,163092,10180
3,394299,251,"Abilene, TX",msa,TX,2010-04-30,107161.740820,2010,42286,7112,15.8,5.6,163092,10180
4,394299,251,"Abilene, TX",msa,TX,2010-05-31,107188.447158,2010,42286,7112,15.8,5.6,163092,10180
